# Notes


LunarLander:

States: 8 continuous values
Actions: 4 discrete
Solved: average reward $\geq$ 200 over 100 episodes

# -

In [ ]:
# TEMP ENV SETUP
# Change to actual env when it's ready

import gymnasium as gym
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical
import matplotlib.pyplot as plt

env = gym.make("LunarLander-v3")
state, _ = env.reset()
print("State shape:", state.shape)   
print("Action space:", env.action_space.n) 

State shape: (8,)
Action space: 4


In [ ]:
class PolicyNetwork(nn.Module):
    def __init__(self, state_dim=8, action_dim=4, hidden_dim=128):
        super(PolicyNetwork, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(state_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, action_dim)
        )
    
    def forward(self, x):
        logits = self.network(x)
        return Categorical(logits=logits)  # returns a distribution

In [ ]:
def select_action(policy, state):
    state_tensor = torch.FloatTensor(state)
    dist = policy(state_tensor)
    action = dist.sample()                    # sample from distribution
    log_prob = dist.log_prob(action)          # save this for the loss later
    return action.item(), log_prob

In [ ]:
def compute_returns(rewards, gamma=0.99):
    returns = []
    G = 0
    for reward in reversed(rewards):
        G = reward + gamma * G
        returns.insert(0, G)
    
    returns = torch.FloatTensor(returns)
    
    # Normalize to reduce variance
    returns = (returns - returns.mean()) / (returns.std() + 1e-8)
    return returns

In [ ]:
def update_policy(optimizer, log_probs, returns):
    loss = []
    for log_prob, G in zip(log_probs, returns):
        loss.append(-log_prob * G)       # negative because we MAXIMIZE reward
    
    loss = torch.stack(loss).sum()
    
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    
    return loss.item()

In [ ]:
def train_reinforce(num_episodes=3000, gamma=0.99, lr=3e-4):
    env = gym.make("LunarLander-v3")
    policy = PolicyNetwork(state_dim=8, action_dim=4, hidden_dim=128)
    optimizer = optim.Adam(policy.parameters(), lr=lr)
    
    rewards_per_episode = []   # hand this to Person 5

    for episode in range(num_episodes):
        state, _ = env.reset()
        log_probs = []
        rewards = []
        done = False

        # --- Collect one episode ---
        while not done:
            action, log_prob = select_action(policy, state)
            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated

            log_probs.append(log_prob)
            rewards.append(reward)
            state = next_state

        # --- After episode ends ---
        returns = compute_returns(rewards, gamma)
        loss = update_policy(optimizer, log_probs, returns)

        total_reward = sum(rewards)
        rewards_per_episode.append(total_reward)

        if (episode + 1) % 100 == 0:
            avg = np.mean(rewards_per_episode[-100:])
            print(f"Episode {episode+1} | Avg Reward (last 100): {avg:.2f} | Loss: {loss:.4f}")

    env.close()
    return policy, rewards_per_episode

# Run it
policy, rewards_per_episode = train_reinforce()

Environment fully solved at $\sim$ 2600-3000 episodes 

In [ ]:
# Save reward log for visualization
np.save("reinforce_rewards.npy", np.array(rewards_per_episode))

# Save model
torch.save(policy.state_dict(), "reinforce_policy.pth")